# NullVector Progress Notebook

Phase: `major-changes-v2` / Phase F-J storage-backed runtime verification.

This notebook exercises acquisition, tree build, unified gateway text usage, unified gateway visual-enrichment usage, and result inspection. It runs against the filesystem backend by default and switches to PostgreSQL when `NULLVECTOR_PROGRESS_POSTGRES_CONNINFO` is set and reachable.

The current pass also validates direct store-backed tree persistence for committed outputs.


### Environment
This notebook prepares a clean artifact root, imports the current runtime surface, and then runs deterministic smoke tests against local fixture PDFs.


In [ ]:
# environment setup
from __future__ import annotations

import json
import os
import platform
import shutil
from pathlib import Path

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "progress-storage-integration"
if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print(
    {
        "python": platform.python_version(),
        "repo_root": str(REPO_ROOT),
        "artifact_root": str(ARTIFACT_ROOT),
    }
)

In [ ]:
# imports
from pydantic import BaseModel

from nullvector.domain import AcquisitionRequest, TreeBuildRequest
from nullvector.domain.common import GeometryCoordinateSpace
from nullvector.domain.tree import VisualEnrichmentRequest, VisualRegionReference
from nullvector.ingest import acquire_document
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayRequest,
    GatewayService,
    LLMMessage,
    LLMRole,
    NoopProviderAdapter,
    NoopScriptedResponse,
    enrich_visual_region,
)
from nullvector.storage import (
    PostgresStorageConfig,
    build_document_store,
    build_postgres_artifact_ref,
)
from nullvector.tree import build_tree

In [ ]:
# configuration
BORN_DIGITAL_PDF = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "born_digital_with_outline.pdf"
VISUAL_PDF = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "mixed_content.pdf"

BORN_DIGITAL_RUN_ID = "progress-storage-acquisition"
BORN_DIGITAL_TREE_RUN_ID = "progress-storage-tree"
VISUAL_RUN_ID = "progress-storage-visual-acquisition"
POSTGRES_CONNINFO = os.environ.get("NULLVECTOR_PROGRESS_POSTGRES_CONNINFO")
postgres_status = None
storage_config = None
if POSTGRES_CONNINFO:
    try:
        import psycopg

        with psycopg.connect(POSTGRES_CONNINFO):
            pass
        storage_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO)
        postgres_status = "enabled"
    except Exception as exc:
        postgres_status = f"fallback-to-filesystem: {exc}"
        storage_config = None
else:
    postgres_status = "filesystem-only"

artifact_store = build_document_store(storage_config, default_filesystem_root=".")

class NotebookTextResponse(BaseModel):
    summary: str

def manifest_ref(*, run_type: str, run_id: str, document_id: str, artifact_root: str) -> str:
    if storage_config is None:
        return str(Path(artifact_root) / "manifest.json")
    return build_postgres_artifact_ref(
        run_type=run_type,
        run_id=run_id,
        document_id=document_id,
        artifact_path="manifest.json",
    )

def read_json_ref(ref: str) -> dict | list:
    if ref.startswith("pg://"):
        return artifact_store.read_json_artifact(ref)
    return json.loads(Path(ref).read_text(encoding="utf-8"))

gateway = GatewayService(
    GatewayConfig(
        default_model="notebook-noop-model",
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "gateway-audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "notebook_text_demo": NoopScriptedResponse(
                output_json={
                    "summary": "The storage integration lets the same pipeline persist parser, tree, retrieval, and audit artifacts to filesystem refs or PostgreSQL refs."
                }
            ),
            "visual_region_enrichment": NoopScriptedResponse(
                output_json={
                    "insight": {
                        "summary": "The selected region is a rendered document image that should remain interpretive visual evidence rather than authoritative native text.",
                        "labels": ["document_scan", "visual_evidence"],
                        "attributes": {"source": "notebook-demo"},
                        "confidence": 0.95,
                    }
                }
            ),
        }
    ),
    storage=storage_config,
)

assert BORN_DIGITAL_PDF.exists(), BORN_DIGITAL_PDF
assert VISUAL_PDF.exists(), VISUAL_PDF
print(
    {
        "postgres_status": postgres_status,
        "storage_backend": "postgres" if storage_config is not None else "filesystem",
    }
)

In [ ]:
# execution
born_digital_manifest = acquire_document(
    AcquisitionRequest(
        source_path=str(BORN_DIGITAL_PDF),
        acquisition_run_id=BORN_DIGITAL_RUN_ID,
        artifact_root=str(ARTIFACT_ROOT / "acquisition_runs"),
    ),
    storage=storage_config,
)
tree_manifest = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=manifest_ref(
            run_type="acquisition",
            run_id=born_digital_manifest.acquisition_run_id,
            document_id=born_digital_manifest.document_id,
            artifact_root=born_digital_manifest.artifact_root,
        ),
        tree_run_id=BORN_DIGITAL_TREE_RUN_ID,
        summarize=False,
    ),
    storage=storage_config,
)

node_cards = read_json_ref(tree_manifest.node_cards_path)
print(
    {
        "acquisition_document_id": born_digital_manifest.document_id,
        "acquisition_page_count": born_digital_manifest.page_count,
        "tree_committed_node_count": tree_manifest.committed_node_count,
        "tree_titles": [card["title"] for card in node_cards],
    }
)

In [ ]:
# execution
text_result = gateway.invoke(
    GatewayRequest[NotebookTextResponse](
        operation_name="notebook_text_demo",
        messages=(
            LLMMessage(
                role=LLMRole.USER,
                content="Summarize the storage-integration impact in one sentence.",
            ),
        ),
        response_model=NotebookTextResponse,
        idempotency_key="notebook-text-demo",
    )
)
print(
    {
        "provider_name": text_result.provider_name,
        "operation_name": text_result.operation_name,
        "summary": text_result.output.summary,
        "audit_path": text_result.audit_path,
    }
)

In [ ]:
# execution
visual_manifest = acquire_document(
    AcquisitionRequest(
        source_path=str(VISUAL_PDF),
        acquisition_run_id=VISUAL_RUN_ID,
        artifact_root=str(ARTIFACT_ROOT / "acquisition_runs"),
    ),
    storage=storage_config,
)
visual_ledger = read_json_ref(visual_manifest.ledger_path)
selected_region_payload = None
selected_page_index = None
for page in visual_ledger["pages"]:
    for block in page["blocks"]:
        if block["block_type"] in {"visual_artifact", "unresolved_region"}:
            selected_region_payload = block
            selected_page_index = page["page_index"]
            break
    if selected_region_payload is not None:
        break

assert selected_region_payload is not None
assert selected_page_index is not None

visual_region = VisualRegionReference(
    document_id=visual_manifest.document_id,
    page_index=selected_page_index,
    region_id=selected_region_payload.get("visual_id", selected_region_payload.get("region_id")),
    bbox=selected_region_payload["bbox"],
    image_ref=selected_region_payload.get("image_ref"),
    asset_path=selected_region_payload.get("asset_path"),
    page_render_path=selected_region_payload.get("page_render_path"),
    render_dpi=selected_region_payload.get("render_dpi"),
    coordinate_space=GeometryCoordinateSpace(
        selected_region_payload.get("coordinate_space", "unrotated_page")
    ),
    node_id=None,
)
visual_attachment = enrich_visual_region(
    gateway,
    VisualEnrichmentRequest(
        request_id="notebook-visual-demo",
        region=visual_region,
        prompt="Describe the selected visual region conservatively and treat it as interpretive evidence.",
        metadata={"fixture": VISUAL_PDF.name},
    ),
)
print(
    {
        "region_id": visual_attachment.region_id,
        "provider_identity": visual_attachment.provider_identity,
        "confidence": visual_attachment.confidence,
        "summary": visual_attachment.insight.summary,
        "audit_path": visual_attachment.audit_path,
    }
)

In [ ]:
# inspect results
verification_report = read_json_ref(tree_manifest.verification_report_path)
strategy_report = read_json_ref(
    tree_manifest.strategy_execution_report_path or tree_manifest.build_report_path
)
summary = {
    "tree_manifest": {
        "artifact_root": tree_manifest.artifact_root,
        "committed_node_count": tree_manifest.committed_node_count,
        "unassigned_span_count": tree_manifest.unassigned_span_count,
    },
    "strategy_report": {
        "attempted_strategies": strategy_report.get("attempted_strategies", []),
        "selected_strategy": strategy_report.get("selected_strategy"),
        "fallback_reasons": strategy_report.get("fallback_reasons", []),
    },
    "verification_status": verification_report["status"],
    "text_gateway_summary": text_result.output.summary,
    "visual_enrichment_summary": visual_attachment.insight.summary,
}
print(json.dumps(summary, indent=2, sort_keys=True))

### Known Limitations
- The notebook exercises real acquisition and tree-build entrypoints against local fixture PDFs, so PyMuPDF and pypdf must be installed.
- Set `NULLVECTOR_PROGRESS_POSTGRES_CONNINFO` to exercise the PostgreSQL backend; if the connection is unavailable, the notebook falls back to the filesystem backend and prints that status in the configuration cell.
- The gateway path stays deterministic through the noop adapter so the notebook remains stable and lightweight.
- Committed tree outputs now persist directly through the active storage backend; filesystem runs emit absolute paths and PostgreSQL runs emit `pg://...` refs.
